# Human-in-the-Loop: Cancelli Pre-Azione, Classificazione del Rischio e Registrazione Audit

Il README di questa lezione introduce Human-in-the-Loop con un breve snippet che chiede all’utente di `APPROVE` o `REJECT` dopo che l’agente ha già prodotto una risposta. Questo schema è un buon punto di partenza, ma nelle implementazioni HITL in produzione sono comunemente necessari tre elementi aggiuntivi:

1. Un **cancello pre-azione** che viene eseguito **prima** che l’agente compia un passo rischioso, così che costo, irreversibilità e latenza rimangano sotto controllo.
2. La **classificazione del rischio**, così le azioni a basso rischio si eseguono automaticamente, quelle a rischio medio sono approvate in batch, e solo le azioni ad alto rischio richiedono il blocco su un umano.
3. Un **registro di audit più un ciclo di revisione**, così ogni decisione del cancello viene registrata come JSONL, e un rifiuto ri-prompta l’agente con una ragione strutturata invece di stampare semplicemente `Revising...`.

Questo notebook costruisce ciascuno di questi elementi sopra gli stessi primitivi di `06-system-message-framework.ipynb`. Viene eseguito end-to-end in `DEMO_MODE = True` (non serve input interattivo) oppure con prompt reali `input()` quando `DEMO_MODE = False`. Nota: in DEMO_MODE il retry del terzo obiettivo è scriptato così le meccaniche del ciclo sono visibili end-to-end. Una reale riclassificazione guidata dalla revisione richiede `DEMO_MODE = False` e un operatore.

**Fuori dal campo di applicazione (gestito in altre lezioni):** autenticazione e controllo degli accessi (minaccia 2 del README della Lezione 06), middleware per chiamate a strumenti (approfondimento MAF Lezione 14), schemi di dibattito multi-agente.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Cancello pre-azione

Lo snippet HITL del README chiama prima l'agente, quindi chiede all'utente di approvare il risultato. Questo è un flusso **post-azione**. L'agente ha già eseguito, quindi il costo della chiamata LLM è già stato pagato, e qualsiasi effetto collaterale (email inviata, riga del database scritta, commento pubblicato) è già accaduto.

Un flusso **pre-azione** inserisce il cancello prima che l'agente esegua il passaggio rischioso. L'agente propone l'azione, il cancello decide se eseguirla, e solo su approvazione l'effetto collaterale si verifica.

| Aspetto | Approvazione post-azione (snippet README) | Cancello pre-azione (questo notebook) |
|---|---|---|
| Quando viene eseguita l'approvazione? | Dopo che l'agente ha prodotto il risultato | Prima che qualsiasi effetto collaterale venga eseguito |
| Costo LLM in caso di rifiuto | Già pagato | Pagato solo per la proposta, non per l'azione |
| Effetti collaterali irreversibili | Possibili (l'azione è già avvenuta) | Prevenuti |
| Chiarezza dell'audit | L'approvazione è una stampa a video | L'approvazione è un record JSON con timestamp, azione, motivo |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Classificazione del rischio

Non ogni azione richiede l'approvazione umana. Una ricerca in sola lettura contro un'API pubblica ha rischi diversi rispetto all'invio di un'email a un cliente. Trattare entrambe allo stesso modo spreca l'attenzione dell'operatore e rallenta l'agente.

Un modello semplice a 3 livelli:

| Livello | Esempi | Flusso di approvazione |
|---|---|---|
| `basso` (sola lettura) | Ricerca in una base di conoscenza, consultare opzioni di volo, recuperare una pagina web pubblica | Esecuzione automatica, registrata per audit |
| `medio` (mutazioni poco costose) | Memorizzare in cache un risultato, incrementare un contatore, programmare un promemoria | Esecuzione automatica, ma revisione giornaliera raggruppata |
| `alto` (rivolto all'esterno o irreversibile) | Inviare un’email, addebitare una carta, pubblicare su un canale pubblico | Blocco in attesa di approvazione umana |

Questa è una possibile classificazione. I sistemi di produzione spesso usano livelli più granulati (ad es., livelli di permessi AWS IAM, livelli di accesso basati su ruoli). La versione a 3 livelli qui sotto è la versione minima utile per un agente che combina azioni di sola lettura e con effetti collaterali.

Il classificatore sottostante usa euristiche basate su parole chiave così che la demo rimanga deterministica e poco costosa. In un sistema di produzione si sostituirebbe questo con un classificatore apprendente o un motore di policy.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Registro di audit e ciclo di revisione

Un `print("Response approved.")` non è un registro di audit. Per la fiducia, ogni decisione del gate dovrebbe essere registrata come un evento strutturato che puoi successivamente interrogare, riprodurre o allegare a una revisione di incidente.

Due elementi:

1. **JSONL append-only.** Una riga per decisione, con timestamp, azione, livello, decisione, motivo. Facile da grep, facile da inviare a un vero archivio di log in seguito.
2. **Ciclo di revisione in caso di rifiuto.** Quando il gate restituisce `deny`, l'agente si rilancia con il motivo del rifiuto nel contesto, così la proposta successiva può evitare il problema.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Risorse aggiuntive

Diversi altri progetti pubblici implementano variazioni di questi modelli HITL. Confronta gli approcci per trovare quello che si adatta al tuo stack:

- **LangChain** wrapper per strumenti human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): wrapper per strumenti plug-and-play che mettono in pausa l'esecuzione per l'input umano.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ha ristrutturato questo): utilizza un ruolo di agente specifico per rappresentare l'umano nelle conversazioni multi-agente.
- **Semantic Kernel** filtri per le funzioni ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware che viene eseguito attorno a ogni chiamata di funzione, adatto per la logica di controllo.

Ogni progetto gestisce i tre sotto-modelli in modo diverso: LangChain li incapsula come strumenti, AutoGen utilizza un ruolo agente, Semantic Kernel usa filtri middleware. Leggi una o due implementazioni dall'inizio alla fine prima di scegliere un design per il tuo agente.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
